1. Comando para clonar o git hub

In [1]:
%%bash

git clone https://github.com/renatopuga/lmabrasil-hg38.git

Cloning into 'lmabrasil-hg38'...


2. Usar o comando para organizar o dataframe, deixando ele compativel com o CGI

In [5]:
%%bash

OUTPUT="df_WPALL-cgi.txt"


FIRST_FILE=$(ls /content/lmabrasil-hg38/vep_output/liftOver_WP*_hg19ToHg38.vep.filter.tsv | head -n 1)

cut -f1-4 "$FIRST_FILE" | head -n 1 | sed -e "s/CHROM/CHR/g" | awk '{print $0"\tSAMPLE"}' > "$OUTPUT"

# 2. PROCESSAR E JUNTAR OS DADOS

for FILE in /content/lmabrasil-hg38/vep_output/liftOver_WP*_hg19ToHg38.vep.filter.tsv; do

    # Extrai o ID da amostra (ex: WP017) do nome do arquivo usando grep
    SAMPLE_ID=$(basename "$FILE" | grep -o "WP[0-9]\+")

    cut -f1-4 "$FILE" | tail -n +2 | awk -v id="$SAMPLE_ID" -v OFS="\t" '{print $0, id}' >> "$OUTPUT"

done

# 3. VERIFICAÇÃO
echo "Arquivo gerado: $OUTPUT"
echo "Primeiras 5 linhas:"
head -n 5 "$OUTPUT"
echo "----------------"
echo "Contagem de linhas por amostra no arquivo final:"
# Isso mostra quantas linhas de cada WP ficaram no arquivo final
cut -f5 "$OUTPUT" | sort | uniq -c


Arquivo gerado: df_WPALL-cgi.txt
Primeiras 5 linhas:
CHR	POS	REF	ALT	SAMPLE
chr1	114716127	C	T	WP017
chr1	152304661	G	C	WP017
chr11	115209621	G	A	WP017
chr12	132140028	C	T	WP017
----------------
Contagem de linhas por amostra no arquivo final:
      1 SAMPLE
     22 WP017
     14 WP019
      2 WP048
      7 WP058
     14 WP068


3. Enviando um request "requisição" a API do CGI
- Esse código em python permite realizar a requisição a API via código **PYTHON**, também tem a opção via bash, mas para fins didaticos, sera explicado via python

In [6]:
import requests
# cabeçalho
headers = {'Authorization': 'kaneka4850@gmail.com c617eeb5f78dc078813f'}
payload = {'cancer_type': 'HEMATO', 'title': 'Somatic ALL', 'reference': 'hg38'}
# requisição
r = requests.post('https://www.cancergenomeinterpreter.org/api/v1',
                headers=headers,
                files={
                        'mutations': open('/content/df_WPALL-cgi.txt', 'rb')
                        },
                data=payload)
r.json() # Formato json para verificação

'bd909eb4c21b4fcee9af'

4. Verificação do Status do Job_ID, lembrando que o job_id é individual

In [7]:
import requests

job_id = input("Digite seu job_id (Obtido na 3º célula)")
headers = {'Authorization': 'kaneka4850@gmail.com c617eeb5f78dc078813f'}
r = requests.get('https://www.cancergenomeinterpreter.org/api/v1/%s' % job_id, headers=headers)
r.json()

Digite seu job_id (Obtido na 3º célula)bd909eb4c21b4fcee9af


{'status': 'Done',
 'metadata': {'id': 'bd909eb4c21b4fcee9af',
  'user': 'kaneka4850@gmail.com',
  'title': 'Somatic ALL',
  'cancertype': 'HEMATO',
  'reference': 'hg38',
  'dataset': 'input.tsv',
  'date': '2025-12-06 23:25:42'}}

5. Verificação das logs

In [8]:
import requests
job_id = input("Digite seu job_id (Obtido na 3º célula)")

headers = {'Authorization': 'kaneka4850@gmail.com c617eeb5f78dc078813f'}
payload={'action':'logs'}
r = requests.get('https://www.cancergenomeinterpreter.org/api/v1/%s' % job_id, headers=headers, params=payload)
r.json()

Digite seu job_id (Obtido na 3º célula)bd909eb4c21b4fcee9af


{'status': 'Done',
 'logs': ['# cgi analyze input.tsv -c HEMATO -g hg38',
  '2025-12-07 00:25:47,764 INFO     Parsing input01.tsv\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping variant with invalid ref/alt: "C/T,A" | Alteration ID: input01_19\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping variant with invalid ref/alt: "-/,GTT,GTT" | Alteration ID: input01_21\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping variant with invalid ref/alt: "A/G,AGG" | Alteration ID: input01_26\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping variant with invalid ref/alt: "-/,GTT" | Alteration ID: input01_27\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping variant with invalid ref/alt: "-/,AT" | Alteration ID: input01_30\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping variant with invalid ref/alt: "C/AA,A" | Alteration ID: input01_31\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping variant with invalid ref/alt: "-/CG,G" | Alteration ID: input01_32\n',
  '2025-12-07 00:25:47,773 WARNING  Skipping varian

6. Fazendo o Download dos arquivos

In [9]:
import requests
job_id = input("Digite seu job_id (Obtido na 3º célula)")

headers = {'Authorization': 'kaneka4850@gmail.com c617eeb5f78dc078813f'} # permissões do CGI
payload={'action':'download'} # passando o que é pra ele fazer
r = requests.get('https://www.cancergenomeinterpreter.org/api/v1/%s' % job_id, headers=headers, params=payload) # requisições
with open('samplefinal.zip', 'wb') as fd:
    fd.write(r._content)

Digite seu job_id (Obtido na 3º célula)bd909eb4c21b4fcee9af


7. Descompactar o .zip utilizando unzip.

In [10]:
%%bash
unzip -o samplefinal.zip

Archive:  samplefinal.zip
  inflating: alterations.tsv         
  inflating: biomarkers.tsv          
  inflating: input01.tsv             
  inflating: summary.txt             


8. Verificando o resultado das alterações: arquivo alterations.tsv

In [11]:
# BAIXA A MERDA DO PANDAS ANTES
import pandas as pd
pd.read_csv('/content/alterations.tsv',sep='\t',index_col=False, engine= 'python')

,Input ID,CHROMOSOME,POSITION,REF,ALT,SAMPLE,CHR,POS,ALT_TYPE,STRAND,...,CGI-Oncogenic Prediction,CGI-External oncogenic annotation,CGI-Mutation,CGI-Consequence,CGI-Transcript,CGI-STRAND,CGI-Type,CGI-HGVS,CGI-HGVSc,CGI-HGVSp
0,input01_1,1,114716127,C,T,WP017,chr1,114716127,snp,+,...,driver (boostDM: non-tissue-specific model),"cgi,clinvar:177778",chr1:114716127 C>T,missense_variant,ENST00000369535,+,SNV,ENST00000369535:c.34G>A;p.(Gly12Ser);p.(G12S),ENST00000369535.5:c.34G>A,ENSP00000358548.4:p.Gly12Ser
1,input01_2,1,152304661,G,C,WP017,chr1,152304661,snp,+,...,passenger (oncodriveMUT),NaN,chr1:152304661 G>C,missense_variant,ENST00000368799,+,SNV,ENST00000368799:c.10225C>G;p.(Arg3409Gly);p.(R...,ENST00000368799.2:c.10225C>G,ENSP00000357789.1:p.Arg3409Gly
2,input01_3,11,115209621,G,A,WP017,chr11,115209621,snp,+,...,passenger (oncodriveMUT),NaN,chr11:115209621 G>A,missense_variant,ENST00000331581,+,SNV,ENST00000331581:c.1031C>T;p.(Thr344Ile);p.(T344I),ENST00000331581.11:c.1031C>T,ENSP00000329797.6:p.Thr344Ile
3,input01_4,12,132140028,C,T,WP017,chr12,132140028,snp,+,...,non-protein affecting,NaN,chr12:132140028 C>T,intron_variant,ENST00000397333,+,SNV,ENST00000397333:c.1775+70G>A;p(?),ENST00000397333.4:c.1775+70G>A,-
4,input01_5,14,24119817,G,A,WP017,chr14,24119817,snp,+,...,passenger (oncodriveMUT),NaN,chr14:24119817 G>A,missense_variant,ENST00000446197,+,SNV,ENST00000446197:c.1013G>A;p.(Arg338His);p.(R338H),ENST00000446197.8:c.1013G>A,ENSP00000415556.4:p.Arg338His
5,input01_6,14,59727407,G,A,WP017,chr14,59727407,snp,+,...,passenger (oncodriveMUT),NaN,chr14:59727407 G>A,missense_variant,ENST00000267484,+,SNV,ENST00000267484:c.1277C>T;p.(Ala426Val);p.(A426V),ENST00000267484.10:c.1277C>T,ENSP00000267484.5:p.Ala426Val
6,input01_7,15,24675943,G,A,WP017,chr15,24675943,snp,+,...,passenger (oncodriveMUT),NaN,chr15:24675943 G>A,missense_variant,ENST00000329468,+,SNV,ENST00000329468:c.76G>A;p.(Ala26Thr);p.(A26T),ENST00000329468.5:c.76G>A,ENSP00000333735.3:p.Ala26Thr
7,input01_8,16,67616834,G,A,WP017,chr16,67616834,snp,+,...,driver (oncodriveMUT),NaN,chr16:67616834 G>A,missense_variant,ENST00000646076,+,SNV,ENST00000646076:c.1042G>A;p.(Glu348Lys);p.(E348K),ENST00000646076.1:c.1042G>A,ENSP00000494538.1:p.Glu348Lys
8,input01_9,16,71389851,G,A,WP017,chr16,71389851,snp,+,...,passenger (oncodriveMUT),NaN,chr16:71389851 G>A,missense_variant,ENST00000302628,+,SNV,ENST00000302628:c.802G>A;p.(Glu268Lys);p.(E268K),ENST00000302628.9:c.802G>A,ENSP00000307508.4:p.Glu268Lys
9,input01_10,16,74452195,A,C,WP017,chr16,74452195,snp,+,...,non-protein affecting,NaN,chr16:74452195 A>C,intron_variant,ENST00000205061,+,SNV,ENST00000205061:c.3537-64T>G;p(?),ENST00000205061.9:c.3537-64T>G,-


9. Verificando o biomarcador (arquivo biomarkers.tsv)

In [12]:

import pandas as pd
pd.read_csv('/content/biomarkers.tsv',sep='\t',index_col=False, engine= 'python')

,Sample ID,Alterations,Biomarker,Drugs,Diseases,Response,Evidence,Match,Source,BioM,Resist.,Tumor type
0,WP017,NRAS (G12S),"NRAS (12,13,59,61,117,146)",Cetuximab(EGFR mAb inhibitor),Colorectal adenocarcinoma,Resistant,A,NO,PMID:24024839;PMID:20619739;PMID:23325582,complete,NaN,COREAD
1,WP017,NRAS (G12S),"NRAS (12,13,59,61,117,146)",Panitumumab(EGFR mAb inhibitor),Colorectal adenocarcinoma,Resistant,A,NO,FDA guidelines,complete,NaN,COREAD
2,WP017,KRAS wildtype,KRAS wildtype,Panitumumab(EGFR mAb inhibitor),Colorectal adenocarcinoma,Responsive,A,NO,FDA https://www.accessdata.fda.gov/drugsatfda...,complete,65.0,COREAD
3,WP017,KRAS wildtype,KRAS wildtype,Cetuximab(EGFR mAb inhibitor),Colorectal adenocarcinoma,Responsive,A,NO,PMID: 19339720,complete,22.0,COREAD
4,WP017,"EGFR wildtype, ALK wildtype",EGFR wildtype;ALK wildtype,Atezolizumab(PDL1 blocking antibody);Bevacizum...,Non-small cell lung,Responsive,A,NO,FDA https://www.accessdata.fda.gov/drugsatfda_...,complete,NaN,NSCLC
...,...,...,...,...,...,...,...,...,...,...,...,...
318,WP048,NRAS (G13D),NRAS Q61R,Temozolomide,Cutaneous melanoma,Responsive,C,NO,https://civicdb.org/links/evidence_items/23,only alteration type,NaN,CM
319,WP048,NRAS (G13D),NRAS Q61R,Cetuximab,Colorectal adenocarcinoma,Resistant,C,NO,https://civicdb.org/links/evidence_items/2184,only alteration type,NaN,COREAD
320,WP048,PIK3CA wildtype,PIK3CA Wildtype,Aspirin,Colorectal adenocarcinoma,Responsive,B,NO,https://civicdb.org/links/evidence_items/2038,complete,NaN,COREAD
321,WP048,TP53 wildtype,TP53 Wildtype,"Cetuximab,Oxaliplatin,Capecitabine",Colorectal adenocarcinoma,Responsive,B,NO,https://civicdb.org/links/evidence_items/875,complete,NaN,COREAD


Extra: gerador de HTML para deixar bonitinho

In [13]:
import pandas as pd
import json
from google.colab import files

# 1. Carregamento dos Dados com Pandas
try:
    df_alt = pd.read_csv('alterations.tsv', sep='\t')
    df_bio = pd.read_csv('biomarkers.tsv', sep='\t')

    print("✅ Arquivos carregados com sucesso!")
except FileNotFoundError:
    print("❌ Erro: Por favor, faça o upload dos arquivos")
    raise

# 2. Limpeza e Pré-processamento
# Substitui valores vazios (NaN) por string vazia para compatibilidade com JSON/HTML
df_alt = df_alt.fillna('')
df_bio = df_bio.fillna('')


# 3. Conversão para JSON
# O Pandas faz isso nativamente, facilitando muito a injeção no HTML
json_alt = df_alt.to_json(orient='records')
json_bio = df_bio.to_json(orient='records')

# 4. Template HTML (A estrutura visual do relatório)
html_template = """
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Relatório Genômico (Gerado via Pandas)</title>
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.11.5/css/jquery.dataTables.css">
    <style>
        :root { --bg: #f4f7f6; --card: #fff; --text: #333; --accent: #3498db; }
        [data-theme="dark"] { --bg: #1a1a1a; --card: #2d2d2d; --text: #e0e0e0; }
        body { font-family: sans-serif; background: var(--bg); color: var(--text); padding: 20px; }
        .container { max-width: 1400px; margin: 0 auto; }
        .card { background: var(--card); padding: 20px; border-radius: 8px; margin-bottom: 20px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); }
        h2 { border-left: 4px solid var(--accent); padding-left: 10px; margin-top: 0; }

        /* Badges */
        .badge { padding: 4px 8px; border-radius: 4px; font-weight: bold; font-size: 0.85em; }
        .oncogenic { background: #ffebee; color: #c0392b; border: 1px solid #c0392b; }
        .passenger { background: #f0f2f5; color: #7f8c8d; border: 1px solid #bdc3c7; }
        .responsive { background: #e8f5e9; color: #27ae60; border: 1px solid #27ae60; }
        .resistant { background: #ffebee; color: #c0392b; border: 1px solid #c0392b; }
        .evidence { display: inline-block; width: 25px; height: 25px; text-align: center; background: var(--accent); color: white; border-radius: 50%; line-height: 25px; }

        /* Controls */
        .controls { display: flex; gap: 15px; margin-bottom: 20px; align-items: center; background: var(--card); padding: 15px; border-radius: 8px; }
        select { padding: 8px; }

        /* Dark Mode overrides for DataTables */
        [data-theme="dark"] .dataTables_wrapper { color: #a0a0a0; }
        [data-theme="dark"] table.dataTable tbody tr { background-color: var(--card); }
    </style>
</head>
<body>

<div class="container">
    <div class="controls">
        <h1>🧬 Dashboard Genômico</h1>
        <select id="sampleSelect"><option value="">Todas as Amostras</option></select>
        <label><input type="checkbox" id="relevantOnly"> Apenas Relevantes</label>
        <button onclick="toggleTheme()">Alternar Tema</button>
    </div>

    <div class="card">
        <h2>🔬 Variantes (Alterations)</h2>
        <table id="altTable" class="display" style="width:100%">
            <thead><tr><th>Sample</th><th>Gene</th><th>Protein</th><th>Type</th><th>Summary</th><th>Prediction</th></tr></thead>
        </table>
    </div>

    <div class="card">
        <h2>💊 Biomarcadores (Biomarkers)</h2>
        <table id="bioTable" class="display" style="width:100%">
            <thead><tr><th>Sample</th><th>Alteration</th><th>Drug</th><th>Disease</th><th>Response</th><th>Evidence</th></tr></thead>
        </table>
    </div>
</div>

<script src="https://code.jquery.com/jquery-3.6.0.min.js"></script>
<script src="https://cdn.datatables.net/1.11.5/js/jquery.dataTables.js"></script>
<script>
    // DADOS INJETADOS PELO PYTHON/PANDAS
    const altData = __ALT_JSON__;
    const bioData = __BIO_JSON__;

    let altTable, bioTable;

    function init() {
        // Popular Select
        const samples = [...new Set([...altData.map(d => d['SAMPLE']), ...bioData.map(d => d['Sample ID'])])].sort();
        const select = document.getElementById('sampleSelect');
        samples.forEach(s => { if(s) select.innerHTML += `<option value="${s}">${s}</option>`; });

        // Tabela Variantes
        altTable = $('#altTable').DataTable({
            data: altData,
            columns: [
                { data: 'SAMPLE' }, { data: 'CGI-Gene' }, { data: 'CGI-Protein Change' }, { data: 'CGI-Type' },
                { data: 'CGI-Oncogenic Summary', render: d => {
                    const isOnco = (d||'').toLowerCase().includes('oncogenic') && !(d||'').toLowerCase().includes('non-oncogenic');
                    return `<span class="badge ${isOnco ? 'oncogenic' : 'passenger'}">${d}</span>`;
                }},
                { data: 'CGI-Oncogenic Prediction' }
            ]
        });

        // Tabela Biomarcadores
        bioTable = $('#bioTable').DataTable({
            data: bioData,
            columns: [
                { data: 'Sample ID' }, { data: 'Alterations' }, { data: 'Drugs' }, { data: 'Diseases' },
                { data: 'Response', render: d => {
                    const isRes = (d||'').toLowerCase().includes('resistant');
                    return `<span class="badge ${isRes ? 'resistant' : 'responsive'}">${d}</span>`;
                }},
                { data: 'Evidence', render: d => `<span class="evidence">${d}</span>` }
            ]
        });

        // Listeners
        $('#sampleSelect, #relevantOnly').on('change', () => { altTable.draw(); bioTable.draw(); });

        // Filtro Customizado
        $.fn.dataTable.ext.search.push((settings, data, idx, row) => {
            const sample = $('#sampleSelect').val();
            const relevant = $('#relevantOnly').is(':checked');

            // Filtro de Amostra
            const rowSample = settings.nTable.id === 'altTable' ? row['SAMPLE'] : row['Sample ID'];
            if (sample && rowSample !== sample) return false;

            // Filtro de Relevância
            if (relevant) {
                if (settings.nTable.id === 'altTable') {
                    const summ = (row['CGI-Oncogenic Summary']||'').toLowerCase();
                    const pred = (row['CGI-Oncogenic Prediction']||'').toLowerCase();
                    if (!pred.includes('driver') && !(summ.includes('oncogenic') && !summ.includes('non-oncogenic'))) return false;
                } else {
                    if (row['Evidence'] !== 'A' && row['Evidence'] !== 'B') return false;
                }
            }
            return true;
        });
    }

    function toggleTheme() {
        const current = document.documentElement.getAttribute('data-theme');
        document.documentElement.setAttribute('data-theme', current === 'dark' ? 'light' : 'dark');
    }

    $(document).ready(init);
</script>
</body>
</html>
"""

# 5. Injeção e Salvamento
# Substituímos os placeholders __ALT_JSON__ e __BIO_JSON__ pelos dados reais do Pandas
final_html = html_template.replace('__ALT_JSON__', json_alt).replace('__BIO_JSON__', json_bio)

output_filename = 'relatorio_pandas_genomica.html'
with open(output_filename, 'w', encoding='utf-8') as f:
    f.write(final_html)

print(f"✅ Arquivo '{output_filename}' gerado com sucesso!")

# 6. Download Automático
files.download(output_filename)

✅ Arquivos carregados com sucesso!
✅ Arquivo 'relatorio_pandas_genomica.html' gerado com sucesso!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

10. Baixar o Google Drive, pra poder salvar os arquivos.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


11. Por fim, salvar todos os arquivos em uma aba no Drive, pra ficar bonitinho

In [16]:
%%bash
MeuDrive="/content/drive/MyDrive/PGBIOAGMAV/Somaticos/AulaPratica"
mkdir -p $MeuDrive # Criar a pastinha no Drive
cp /content/alterations.tsv $MeuDrive #alterações
cp /content/biomarkers.tsv $MeuDrive # biomarcadores
cp /content/df_WPALL-cgi.txt $MeuDrive # amostra
cp /content/input01.tsv $MeuDrive # id
cp /content/samplefinal.zip $MeuDrive # amostra
cp /content/summary.txt $MeuDrive # sumario

12. Deletando o Job do CGI

In [ ]:
import requests
 # job_id ="314a1ffb93d1bb435a3e"  id individual

headers = {'Authorization': 'kaneka4850@gmail.com c617eeb5f78dc078813f'} # permissões do CGI
r = requests.delete('https://www.cancergenomeinterpreter.org/api/v1/%s' % job_id, headers=headers)
r.json()

print("Job deletado com sucesso")